# AM01 AAE Warm-up Smooth L1 Experiment

Notebook sperimentale mirato all'ultima ablation AAE promettente: **Smooth L1 + warm-up ricostruttivo + ramp lineare della componente adversarial**.

Direzione testata: l'AAE potrebbe non migliorare AE perche' il latent space viene regolarizzato verso `N(0,I)` troppo presto, quando encoder e decoder non hanno ancora imparato una manifold normale stabile.

Questo notebook non modifica il protocollo ufficiale: salva tutto in una cartella separata e serve a decidere se il warm-up merita di entrare nel report finale.

## 0. Parametri

In [ ]:
from pathlib import Path

REPO_URL = ""  # opzionale: https://github.com/<user>/<repo>.git
PROJECT_DIR = Path('/content/am01-kuka-aae-anomaly-detection')
DRIVE_ROOT = Path('/content/drive/MyDrive/AM01')
DATA_DIR = DRIVE_ROOT / 'data' / 'KukaVelocityDataset'
RESULTS_ROOT = DRIVE_ROOT / 'results' / 'official' / 'warmup_aae_smoothl1'
TABLES_DIR = RESULTS_ROOT / 'tables'
FIGURES_DIR = RESULTS_ROOT / 'figures'
RUNS_DIR = RESULTS_ROOT / 'runs'
CONFIG_DIR = RESULTS_ROOT / 'config'

RUN_BASELINES = True
RUN_WARMUP_GRID = True
RUN_CONFIRMATION_MULTI_SEED = True

PRIMARY_WINDOW_LENGTH = 64
PRIMARY_STRIDE = 16
SEED_EXPLORATION = 42
CONFIRMATION_SEEDS = [0, 1, 2]
SELECTION_METRIC = 'val_pr_auc'

# Griglia minima: niente w128, niente tuning esteso.
WARMUP_CONFIGS = [
    {'name': 'aae_smoothl1_no_warmup', 'warmup_epochs': 0, 'ramp_epochs': 0, 'lambda_adv': 0.10},
    {'name': 'aae_smoothl1_warm5_ramp5_lam0p10', 'warmup_epochs': 5, 'ramp_epochs': 5, 'lambda_adv': 0.10},
    {'name': 'aae_smoothl1_warm10_ramp5_lam0p10', 'warmup_epochs': 10, 'ramp_epochs': 5, 'lambda_adv': 0.10},
    {'name': 'aae_smoothl1_warm10_ramp5_lam0p01', 'warmup_epochs': 10, 'ramp_epochs': 5, 'lambda_adv': 0.01},
    {'name': 'aae_smoothl1_warm10_ramp5_lam0p05', 'warmup_epochs': 10, 'ramp_epochs': 5, 'lambda_adv': 0.05},
]

print('DATA_DIR:', DATA_DIR)
print('RESULTS_ROOT:', RESULTS_ROOT)

## 1. Setup Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
import sys


def sh(cmd: str) -> None:
    print(f"\n$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

for path in [RESULTS_ROOT, TABLES_DIR, FIGURES_DIR, RUNS_DIR, CONFIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not PROJECT_DIR.exists():
    if not REPO_URL:
        raise RuntimeError('PROJECT_DIR non esiste. Imposta REPO_URL o carica il repo in /content.')
    sh(f'git clone {REPO_URL} "{PROJECT_DIR}"')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
print('Working directory:', Path.cwd())

sh('pip install -q -r requirements.txt')
sh('python -m compileall -q src scripts')
sh(f'pip freeze > "{CONFIG_DIR / "environment.txt"}"')

## 2. Import e utility

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

import am01.reporting as rpt

sns.set_theme(style='whitegrid', context='notebook')


def run_experiment_command(config, output, *, seed=42, loss='huber', lambda_adv=None, warmup=None, ramp=None, configs='configs/aae_mlp.yaml'):
    parts = [
        'python scripts/run_experiments.py',
        f'--configs {configs}',
        f'--data "{DATA_DIR}"',
        f'--output "{output}"',
        f'--seeds {seed}',
        f'--window-lengths {PRIMARY_WINDOW_LENGTH}',
        f'--losses {loss}',
        '--skip-existing',
        f'--summary-name {config}.csv',
    ]
    if lambda_adv is not None:
        parts.append(f'--lambda-advs {lambda_adv}')
    if warmup is not None:
        parts.append(f'--warmup-epochs {warmup}')
    if ramp is not None:
        parts.append(f'--ramp-epochs {ramp}')
    sh(' '.join(parts))


def collect(root):
    df = rpt.collect_run_metrics(root)
    df.to_csv(TABLES_DIR / f'{Path(root).name}_metrics.csv', index=False)
    return df

## 3. Baseline Smooth L1

Confronto minimo: Isolation Forest, AE + Smooth L1 e AAE + Smooth L1 senza warm-up.

In [ ]:
baseline_root = RUNS_DIR / 'baselines_seed42'
if RUN_BASELINES:
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/isolation_forest.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{baseline_root}" '
        f'--seeds {SEED_EXPLORATION} '
        f'--window-lengths {PRIMARY_WINDOW_LENGTH} '
        f'--skip-existing '
        f'--summary-name isolation_forest.csv'
    )
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/ae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{baseline_root}" '
        f'--seeds {SEED_EXPLORATION} '
        f'--window-lengths {PRIMARY_WINDOW_LENGTH} '
        f'--losses huber '
        f'--skip-existing '
        f'--summary-name ae_smoothl1.csv'
    )
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{baseline_root}" '
        f'--seeds {SEED_EXPLORATION} '
        f'--window-lengths {PRIMARY_WINDOW_LENGTH} '
        f'--losses huber '
        f'--lambda-advs 0.1 '
        f'--warmup-epochs 0 '
        f'--ramp-epochs 0 '
        f'--skip-existing '
        f'--summary-name aae_smoothl1_no_warmup.csv'
    )

baseline_metrics = collect(baseline_root)
display_cols = [c for c in ['model', 'run_name', 'seed', 'loss', 'lambda_adv', 'warmup_epochs', 'ramp_epochs', 'val_pr_auc', 'test_pr_auc', 'test_f1', 'test_roc_auc', 'test_false_alarms_per_run'] if c in baseline_metrics]
display(baseline_metrics[display_cols].sort_values('test_pr_auc', ascending=False).style.format(precision=4))

## 4. Warm-up grid AAE Smooth L1

Tutti i run usano `window_length=64`, `StandardScaler`, `Smooth L1`, `latent_dim=16`.

In [ ]:
warmup_root = RUNS_DIR / 'warmup_grid_seed42'
if RUN_WARMUP_GRID:
    for cfg in WARMUP_CONFIGS:
        run_experiment_command(
            cfg['name'],
            warmup_root,
            seed=SEED_EXPLORATION,
            loss='huber',
            lambda_adv=cfg['lambda_adv'],
            warmup=cfg['warmup_epochs'],
            ramp=cfg['ramp_epochs'],
            configs='configs/aae_mlp.yaml',
        )

warmup_metrics = collect(warmup_root)
warmup_metrics.to_csv(TABLES_DIR / 'warmup_grid_results.csv', index=False)
display_cols = [c for c in ['run_name', 'seed', 'lambda_adv', 'warmup_epochs', 'ramp_epochs', 'val_pr_auc', 'val_f1', 'test_pr_auc', 'test_f1', 'test_roc_auc', 'test_false_alarms_per_run'] if c in warmup_metrics]
display(warmup_metrics[display_cols].sort_values(SELECTION_METRIC, ascending=False).style.format(precision=4))

best_warmup = warmup_metrics.sort_values(SELECTION_METRIC, ascending=False).iloc[0]
BEST_WARMUP = {
    'lambda_adv': float(best_warmup['lambda_adv']),
    'warmup_epochs': int(best_warmup['warmup_epochs']),
    'ramp_epochs': int(best_warmup['ramp_epochs']),
    'run_name': best_warmup['run_name'],
}
display(Markdown(f"**Best warm-up by `{SELECTION_METRIC}`:** `{BEST_WARMUP['run_name']}`"))
BEST_WARMUP

## 5. Figure: warm-up grid

In [ ]:
plot_df = warmup_metrics.copy()
plot_df['schedule'] = plot_df.apply(lambda r: f"warm={int(r['warmup_epochs'])}, ramp={int(r['ramp_epochs'])}, lam={r['lambda_adv']:.3g}", axis=1)

plt.figure(figsize=(11, 4.8))
ax = sns.barplot(data=plot_df.sort_values(SELECTION_METRIC, ascending=False), x='schedule', y='val_pr_auc', color='tab:blue')
ax.set_title('AAE Smooth L1 warm-up grid: validation PR-AUC')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
rpt.savefig(FIGURES_DIR / 'warmup_grid_val_pr_auc.png')
plt.show()

plt.figure(figsize=(11, 4.8))
long = plot_df.melt(id_vars='schedule', value_vars=['val_pr_auc', 'test_pr_auc', 'test_f1'], var_name='metric', value_name='value')
ax = sns.barplot(data=long, x='schedule', y='value', hue='metric')
ax.set_title('AAE Smooth L1 warm-up grid: validation/test metrics')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
rpt.savefig(FIGURES_DIR / 'warmup_grid_metrics.png')
plt.show()

## 6. Training dynamics della migliore configurazione

Verifica se il warm-up produce una transizione stabile: lambda effettiva, reconstruction loss, discriminator/adversarial loss e accuratezza del discriminator.

In [ ]:
best_run_dir = Path(best_warmup['run_dir'])
history_path = best_run_dir / 'history.json'
with history_path.open('r', encoding='utf-8') as f:
    history = json.load(f)

history_df = pd.DataFrame(history)
history_df['epoch'] = np.arange(1, len(history_df) + 1)
history_df.to_csv(TABLES_DIR / 'best_warmup_history.csv', index=False)
display(history_df.tail(10).style.format(precision=4))

plot_cols = [
    'val_loss',
    'discriminator_loss',
    'adversarial_loss',
    'effective_lambda_adv',
    'discriminator_accuracy_real',
    'discriminator_accuracy_fake',
    'latent_mean_norm',
    'latent_covariance_error',
]
existing = [c for c in plot_cols if c in history_df.columns]
long_hist = history_df.melt(id_vars='epoch', value_vars=existing, var_name='signal', value_name='value')

g = sns.FacetGrid(long_hist, col='signal', col_wrap=2, sharey=False, height=3.0)
g.map_dataframe(sns.lineplot, x='epoch', y='value')
g.fig.suptitle('Best warm-up AAE training dynamics', y=1.03)
g.savefig(FIGURES_DIR / 'best_warmup_training_dynamics.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Multi-seed confirmation

Conferma minima su seed `[0, 1, 2]`: Isolation Forest, AE Smooth L1, AAE Smooth L1 senza warm-up, AAE Smooth L1 con migliore warm-up.

In [ ]:
confirm_root = RUNS_DIR / 'confirmation_multiseed'
seed_args = ' '.join(str(s) for s in CONFIRMATION_SEEDS)
if RUN_CONFIRMATION_MULTI_SEED:
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/isolation_forest.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{confirm_root}" '
        f'--seeds {seed_args} '
        f'--window-lengths {PRIMARY_WINDOW_LENGTH} '
        f'--skip-existing '
        f'--summary-name if_multiseed.csv'
    )
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/ae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{confirm_root}" '
        f'--seeds {seed_args} '
        f'--window-lengths {PRIMARY_WINDOW_LENGTH} '
        f'--losses huber '
        f'--skip-existing '
        f'--summary-name ae_smoothl1_multiseed.csv'
    )
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{confirm_root}" '
        f'--seeds {seed_args} '
        f'--window-lengths {PRIMARY_WINDOW_LENGTH} '
        f'--losses huber '
        f'--lambda-advs 0.1 '
        f'--warmup-epochs 0 '
        f'--ramp-epochs 0 '
        f'--skip-existing '
        f'--summary-name aae_smoothl1_no_warmup_multiseed.csv'
    )
    sh(
        f'python scripts/run_experiments.py '
        f'--configs configs/aae_mlp.yaml '
        f'--data "{DATA_DIR}" '
        f'--output "{confirm_root}" '
        f'--seeds {seed_args} '
        f'--window-lengths {PRIMARY_WINDOW_LENGTH} '
        f'--losses huber '
        f'--lambda-advs {BEST_WARMUP["lambda_adv"]} '
        f'--warmup-epochs {BEST_WARMUP["warmup_epochs"]} '
        f'--ramp-epochs {BEST_WARMUP["ramp_epochs"]} '
        f'--skip-existing '
        f'--summary-name aae_smoothl1_best_warmup_multiseed.csv'
    )

confirm_metrics = collect(confirm_root)
confirm_metrics['experiment'] = confirm_metrics.apply(
    lambda r: (
        'Isolation Forest' if r['model_key'] == 'isolation_forest'
        else 'AE Smooth L1' if r['model_key'] == 'ae_mlp'
        else 'AAE Smooth L1 warm-up' if int(r.get('warmup_epochs') or 0) > 0
        else 'AAE Smooth L1 no warm-up'
    ),
    axis=1,
)
confirm_metrics.to_csv(TABLES_DIR / 'confirmation_multiseed_results.csv', index=False)
display(confirm_metrics[['experiment', 'run_name', 'seed', 'lambda_adv', 'warmup_epochs', 'ramp_epochs', 'val_pr_auc', 'test_pr_auc', 'test_f1', 'test_false_alarms_per_run']].sort_values(['experiment', 'seed']).style.format(precision=4))

## 8. Figure: multi-seed confirmation

In [ ]:
plt.figure(figsize=(9, 4.8))
ax = sns.boxplot(data=confirm_metrics, x='experiment', y='test_pr_auc')
sns.stripplot(data=confirm_metrics, x='experiment', y='test_pr_auc', color='black', size=4, ax=ax)
ax.set_title('Multi-seed confirmation: test PR-AUC')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=20)
rpt.savefig(FIGURES_DIR / 'confirmation_multiseed_test_pr_auc.png')
plt.show()

plt.figure(figsize=(9, 4.8))
ax = sns.boxplot(data=confirm_metrics, x='experiment', y='test_f1')
sns.stripplot(data=confirm_metrics, x='experiment', y='test_f1', color='black', size=4, ax=ax)
ax.set_title('Multi-seed confirmation: test F1')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=20)
rpt.savefig(FIGURES_DIR / 'confirmation_multiseed_test_f1.png')
plt.show()

summary = confirm_metrics.groupby('experiment').agg(
    mean_test_pr_auc=('test_pr_auc', 'mean'),
    std_test_pr_auc=('test_pr_auc', 'std'),
    mean_test_f1=('test_f1', 'mean'),
    std_test_f1=('test_f1', 'std'),
    mean_false_alarms_per_run=('test_false_alarms_per_run', 'mean'),
).reset_index().sort_values('mean_test_pr_auc', ascending=False)
summary.to_csv(TABLES_DIR / 'confirmation_multiseed_summary.csv', index=False)
display(summary.style.format(precision=4))

## 9. Sintesi automatica

In [ ]:
best_single = warmup_metrics.sort_values(SELECTION_METRIC, ascending=False).iloc[0]
best_confirm = summary.iloc[0]

ae_rows = confirm_metrics[confirm_metrics['experiment'] == 'AE Smooth L1']
warm_rows = confirm_metrics[confirm_metrics['experiment'] == 'AAE Smooth L1 warm-up']
no_warm_rows = confirm_metrics[confirm_metrics['experiment'] == 'AAE Smooth L1 no warm-up']

warm_vs_ae = warm_rows['test_pr_auc'].mean() - ae_rows['test_pr_auc'].mean() if len(ae_rows) and len(warm_rows) else np.nan
warm_vs_no = warm_rows['test_pr_auc'].mean() - no_warm_rows['test_pr_auc'].mean() if len(no_warm_rows) and len(warm_rows) else np.nan

lines = [
    '# AAE Smooth L1 warm-up experiment summary',
    '',
    f'- Best single-seed warm-up by `{SELECTION_METRIC}`: `{best_single["run_name"]}`.',
    f'- Best warm-up params: warmup={int(best_single["warmup_epochs"])}, ramp={int(best_single["ramp_epochs"])}, lambda_adv={best_single["lambda_adv"]}.',
    f'- Single-seed validation PR-AUC: {best_single["val_pr_auc"]:.4f}; test PR-AUC: {best_single["test_pr_auc"]:.4f}; test F1: {best_single["test_f1"]:.4f}.',
    f'- Best multi-seed mean model by test PR-AUC: `{best_confirm["experiment"]}` with mean PR-AUC={best_confirm["mean_test_pr_auc"]:.4f}.',
    f'- Warm-up AAE minus AE Smooth L1 mean test PR-AUC: {warm_vs_ae:.4f}.',
    f'- Warm-up AAE minus no-warm-up AAE Smooth L1 mean test PR-AUC: {warm_vs_no:.4f}.',
    '',
    'Interpretation guide:',
    '- If warm-up AAE beats AE Smooth L1 across seeds, adversarial regularization is useful after stabilization.',
    '- If warm-up improves AAE but remains below AE, warm-up reduces instability but does not justify the added complexity.',
    '- If warm-up does not improve no-warm-up AAE, the negative conclusion on the adversarial component becomes stronger.',
]
summary_md = '\n'.join(lines)
(RESULTS_ROOT / 'summary.md').write_text(summary_md, encoding='utf-8')
display(Markdown(summary_md))

artifacts = sorted([p for p in RESULTS_ROOT.rglob('*') if p.is_file()])
manifest = pd.DataFrame({'artifact': [str(p.relative_to(RESULTS_ROOT)) for p in artifacts]})
manifest.to_csv(TABLES_DIR / 'artifact_manifest.csv', index=False)
display(manifest)